# Answer Extraction

In [1]:
import json
from pathlib import Path
from evaluator import AnswerExtractor

extractor = AnswerExtractor()

results_base = Path("/home/dario/miniconda3/envs/denv/Personas-Effect-On-Factual-Knowledge-Retrieval/results")
answers_base = Path("/home/dario/miniconda3/envs/denv/Personas-Effect-On-Factual-Knowledge-Retrieval/answers")

fail_count = 0
tot = 0

for input_path in results_base.glob("*/*/*.json"):
    dataset = input_path.parts[-3]
    model = input_path.parts[-2]
    filename = input_path.name

    output_path = answers_base / dataset / model / filename

    # if output_path.exists():
    #     continue

    output_path.parent.mkdir(parents=True, exist_ok=True)

    with input_path.open() as f:
        results = json.load(f)

    runs = []
    for run_results in results:

        items = []
        for item in run_results:
            tot = tot+1
            output = item.get("output", {})
            output_text = output.get("output_text") or output.get("text", "")
            finish_reason = output.get("finish_reason")
    
            if dataset == 'gpqa':
                choices = ['A', 'B', 'C', 'D']
            else:
                choices = item.get("options") or item.get("choices")
    
            golden = item.get("answer")
            question = item.get("question") or item.get("problem")
            subject = item.get("subject") or item.get("category")
            logprobs = output.get("logprobs")

            extracted = extractor.extract(
                output_text=output_text,
                choices=choices,
                dataset=dataset,
                finish_reason=finish_reason,
            )

            answer_logprob, cleaned_logprobs = extractor.get_logprobs(extracted, logprobs)

            if extracted.startswith('N.D.'):
                fail_count = fail_count + 1
                # print(f"## OUTPUT ## :{output_text[len(output_text) // 2 :]}\n\n\n")

            items.append({
                "subject": subject,
                "question": question,
                "choices": choices,
                "golden_label": golden,
                "finish_reason": finish_reason,
                'logprobs': cleaned_logprobs,
                "extracted_answer": extracted,
                "extracted_logprob": answer_logprob,
            })

        runs.append(items)

    output_path.write_text(json.dumps(runs, indent=2, ensure_ascii=False) + "\n")

print(f"tot: {tot} fails: {fail_count} ratio: {fail_count/tot}")

[nltk_data] Downloading package stopwords to /home/dario/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


tot: 7350 fails: 557 ratio: 0.07578231292517007
